# Gold Layer — Dimension: Customer
## SalesFlow Data Lakehouse | Phase 5: Analytical Layer

Reads `salesflow_dev.silver.customers` (VALID records only), builds the
customer dimension with a surrogate key, and writes to `salesflow_dev.gold.dim_customer`.

**Design:**
| Column | Type | Description |
|---|---|---|
| `customer_key` | PK | MD5 surrogate key derived from `CustomerID` |
| `customer_id` | NK | Natural key from source system |
| `company_name` | string | Cleaned company name |
| `contact_name` | string | Primary contact |
| `country` | string | Country |
| `city` | string | City |
| `region` | string | Region (`N/A` if not applicable) |
| `phone` | string | Standardized phone number |
| `effective_date` | date | Date this record was loaded into Gold |

In [0]:
%run ../04_Utils/common_functions

## 1. Read from Silver (VALID records only)

In [0]:
from pyspark.sql.functions import current_date, col

# Read only VALID records from Silver — invalid records must not reach Gold
df = spark.table("salesflow_dev.silver.customers") \
          .filter(col("data_quality_status") == "VALID")

print(f"Valid records read from Silver: {df.count()}")
display(df.limit(5))

## 2. Select and Rename Columns
Rename to snake_case convention used across the Gold layer.

In [0]:
# Select only the columns needed for the dimension and rename to snake_case
df = df.select(
    col("CustomerID").alias("customer_id"),
    col("CompanyName").alias("company_name"),
    col("ContactName").alias("contact_name"),
    col("Country").alias("country"),
    col("City").alias("city"),
    col("Region").alias("region"),
    col("Phone").alias("phone")
)

## 3. Add Surrogate Key
Generates `customer_key` as an MD5 hash of `customer_id`.  
The hash is deterministic — same `customer_id` always produces the same key.

In [0]:
# Add surrogate key based on natural key customer_id
df = add_surrogate_key(df, "customer", ["customer_id"])

## 4. Add Effective Date
Captures the date this dimension record was loaded into Gold.

In [0]:
# effective_date marks when this record entered the Gold layer
df = df.withColumn("effective_date", current_date())

## 5. Final Column Order
Enforce the defined schema order: PK first, NK second, attributes, metadata last.

In [0]:
# Enforce final column order as per dimension design
df = df.select(
    "customer_key",
    "customer_id",
    "company_name",
    "contact_name",
    "country",
    "city",
    "region",
    "phone",
    "effective_date"
)

print(f"Total records in dimension: {df.count()}")
display(df.limit(5))

## 6. Save as Delta Table

In [0]:
# Write to Gold layer as Delta table — overwrite for first load
df.write \
  .format("delta") \
  .mode("overwrite") \
  .saveAsTable("salesflow_dev.gold.dim_customer")

print("Table saved: salesflow_dev.gold.dim_customer")

## 7. Validation

In [0]:
dim_customer = spark.table("salesflow_dev.gold.dim_customer")

# Record count
print(f"Total records: {dim_customer.count()}")

# Surrogate key uniqueness check — must be 0 duplicates
duplicate_keys = dim_customer.groupBy("customer_key").count().filter(col("count") > 1)
print(f"\nDuplicate surrogate keys (expected 0): {duplicate_keys.count()}")

# Country distribution — useful sanity check
print("\nRecords by country:")
display(dim_customer.groupBy("country").count().orderBy(col("count").desc()))

# Schema
print("\nSchema:")
dim_customer.printSchema()

# Sample
print("\nFirst 5 rows:")
display(dim_customer.limit(5))